# v2.04 — v2.03 + news-robust training (validation only)

**One change on top of v2.03** (Siwat, 2026-10-27, `configs/v2_04.json`):
- **What changes:** each training window, with probability **0.2**, gets the no-news fill [⅓, ⅓, ⅓] on every day of its 3 sentiment channels (unit: the whole 126-day window).
- **What stays:** everything else is exactly v2.03.
- **What is scored:** the hold-out (epoch choice) and validation are always scored with the news as it is.

**Why:** v2.03beta's placebo showed that the sentiment content adds nothing, while missing news costs about 0.03 at 5 / 10 days and pushes calls to "down" (`claude/log-entry-2026-10-27-v2-03beta-results.md`).

**Runtime:** A100 or L4 GPU, Colab only (Linux; see `claude/log-entry-2026-10-27-adx-tie-platform.md`). Runtime → Run all.
- **Order:** the three seeds train one after another (parallel gave no speed-up). Then the post-hoc step runs: temperature + blend fitted on v2.04's own inner hold-out, news blanked, placebo.
- **Time:** roughly 1–1.5 h of training on an A100, then about 15–25 min for the post-hoc step.

**Adopt rule** (pre-registered, `claude/log-entry-2026-10-27-v2-04-design.md`), judged on the raw model:
- **(a)** validation with news blanked: 3-horizon mean above v2.03's;
- **(b)** news on: the paired 95% interval of v2.04 − v2.03 is not entirely below 0.

ts + blend are reported beside the raw model.

Outputs: `Senior/results/v2xx/<GPU>/v2_04*`, `Senior/results/v2_04_posthoc/`, `Senior/results/v2_04_vs_v2_03.json`. If Colab disconnects, Run all again: finished stages are skipped. The test block stays sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
GPU_TAG = 'A100' if 'A100' in name else 'L4' if 'L4' in name else None
assert GPU_TAG, 'stop: switch the runtime to an A100 or L4 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, subprocess, sys
from pathlib import Path
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, RES = f'{SENIOR}/prices', f'{SENIOR}/results'
OUT = f'{RES}/v2xx/{GPU_TAG}'                      # one folder per GPU, as v2.01-v2.03
V204 = f'{OUT}/v2_04.json'
POST = f'{RES}/v2_04_posthoc'                      # v2.04's post-hoc step (same script as v2.03beta)
BASE = f'{RES}/v2_03beta'                          # v2.03's post-hoc step (2026-10-27), the comparison
FWD_INPUTS = f'{RES}/forward_2026/inputs'
for f in [f'{BASE}/step0.json', f'{BASE}/trees.csv.gz', f'{BASE}/parts/val_blank.csv.gz', f'{FWD_INPUTS}/forward_prices.csv',
          f'{PRICE_DIR}/set50_union_2019_2025.csv', f'{PRICE_DIR}/yahoo_events_2019_2025.csv']:
    assert os.path.exists(f), f'stop: missing {f}'
os.makedirs(OUT, exist_ok=True)

def stream(args):
    # stream a script's output into the notebook and stop on failure
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        if 'defined on' not in line and 'not writable' not in line and 'from_numpy' not in line:
            print(line, end='', flush=True)
    if p.wait() != 0:
        raise SystemExit(f'stop: {" ".join(args[1:3])} failed (exit {p.returncode}); see the lines above')

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
for f in ['configs/v2_04.json', 'scripts/train_v2xx.py', 'src/train_utils.py', 'src/run_config.py', 'tests/test_news_blank.py',
          'scripts/step0_v2_03beta.py', 'scripts/compare_v2xx_posthoc.py']:
    assert os.path.exists(f), f'stop: push the v2.04 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the frozen copy'
sys.path.insert(0, 'src'); sys.path.insert(0, 'scripts')
import run_config as RC
cfg = RC.load('configs/v2_04.json')
print('v2.04 training:', json.dumps(cfg['training']))

In [ ]:
# Tests (about 6 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_training.py', 'tests/test_news_blank.py', 'tests/test_step0_v2_03beta.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l and '    seed' not in l)[-2500:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 1. Train v2.04: 3 seeds one after another (each epoch prints the hold-out scores, the share of windows blanked and the kept epoch)
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
if done(V204):
    print('v2.04: already trained')
else:
    stream(['python', '-u', 'scripts/train_v2xx.py', '--config', 'configs/v2_04.json', '--out', V204])

In [ ]:
# 2. Summary beside v2.03 and the floors, and the learning curves
r = json.load(open(V204)); s, fl = r['summary']['model'], r['floors']
V203 = {'h1': 0.3418, 'h5': 0.3705, 'h10': 0.3716}; BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
print(f"{'':6}{'v2.04 F1':>18}{'v2.03':>8}{'bar':>7}{'continues':>11}{'reverses':>10}{'R2_OS':>10}")
for h in ('h1', 'h5', 'h10'):
    m = s[h]
    print(f"{h:6}{m['macro_f1']['mean']:>11.4f} ± {m['macro_f1']['std']:.4f}{V203[h]:>8.4f}{BAR[h]:>7.3f}"
          f"{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}{m['r2_os_vs_zero']['mean']:>+10.4f}")
for x in r['runs']:
    print('seed', x['seed'], 'epochs', x['epochs_run'], 'kept', x['best_epoch'], 'hold-out F1', x['best_inner_es_macro_f1'],
          'blanked share (last epoch)', x['history'][-1].get('news_blanked_share'), 'minutes', round(x['seconds'] / 60, 1))
from IPython.display import Image, display
import importlib, plot_v2xx_training as PT
for png in (PT.plot(Path(V204)), PT.plot_calls(Path(V204))):
    if png:
        print('saved', png); display(Image(filename=str(png)))

In [ ]:
# 3. Post-hoc step for v2.04 (same script and settings as v2.03beta): reproduction check, ts, blend (w on v2.04's own
#    inner hold-out), news blanked and placebo, on validation and 2026
common = ['--run-json', V204, '--fwd-inputs', FWD_INPUTS, '--out-dir', POST, '--label', 'v2.04 post-hoc']
for stage in ('prepare', 'predict', 'trees', 'analyze'):
    stream(['python', 'scripts/step0_v2_03beta.py', stage, *common])

In [ ]:
# 4. v2.04 vs v2.03 on the same rows, and the pre-registered adopt rule
stream(['python', 'scripts/compare_v2xx_posthoc.py', '--base', BASE, '--new', POST, '--out', f'{RES}/v2_04_vs_v2_03.json'])

**Reading the result**
- **ADOPT RULE** (last block): (a) the news-blanked 3-horizon mean goes up, and (b) news-on is not significantly below v2.03. Both are judged on the raw models.
- **ts + blend** rows show the scored line (Siwat, 2026-10-27). Raw and XGBoost alone are always beside it.
- **News placebo** for v2.04 is in the post-hoc printout: compare *placebo − blanked* with v2.03's 0.030 / 0.030 at 5 / 10 days. A smaller gap means the model relies less on "news present".
- **2026** is a seen window: reported, never used to adopt.
- Send the printout to Claude.